# 01 — ANSS ComCat rebuild and magnitude harmonisation

Rebuilds the final modelling catalogue.

The API query starts at reported magnitude M≥2.0 so magnitude harmonisation is
performed **before** the final Mw-equivalent threshold is imposed. The final
input catalogue is then filtered to \(M_w^\ast\ge2.5\).

For exact submitted-result reproduction, prefer the archived processed catalogue:
ANSS ComCat is a live catalogue and historical records can be revised.

In [ ]:
from pathlib import Path
from io import StringIO
import time
import warnings

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 100)

# ---------------------------------------------------------
# Project configuration
# ---------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
AUDIT_DIR = OUTPUT_DIR / "audit"

for _d in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR, PRED_DIR, AUDIT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw_usgs_m20"
RAW_DIR.mkdir(parents=True, exist_ok=True)

GRID_MASK_FILE = DATA_DIR / "california_grid_centre_mask.csv"
REFERENCE_M25_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_M25_centre_mask.csv"

RAW_COMBINED_FILE = DATA_DIR / "usgs_bbox_2010_2025_reported_Mge2.csv"
CA_M20_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_reported_Mge2_centre_mask.csv"
HARMONISED_ALL_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_Mw_harmonised_all.csv"
FINAL_MW25_FILE = DATA_DIR / "earthquake_california_grid_2010_2025_Mw25_centre_mask.csv"
UNRESOLVED_FILE = DATA_DIR / "magnitude_harmonisation_unresolved.csv"

# Study period and original bounding box
START_YEAR = 2010
END_YEAR = 2025
LAT_MIN, LAT_MAX = 32.0, 42.5
LON_MIN, LON_MAX = -125.0, -114.0
RAW_MIN_MAG = 2.0

# Final grid definition used by the saved centre-based mask
N_LAT = 53
N_LON = 55
LAT_EDGES = np.linspace(LAT_MIN, LAT_MAX, N_LAT + 1)
LON_EDGES = np.linspace(LON_MIN, LON_MAX, N_LON + 1)

print("Working directory:", DATA_DIR.resolve())
print("Latitude cell width:", LAT_EDGES[1] - LAT_EDGES[0])
print("Longitude cell width:", LON_EDGES[1] - LON_EDGES[0])

## 1. Load and validate the fixed centre-based California grid mask

The spatial domain is **not rebuilt from a shapefile here**. The saved 1,080-cell mask is treated as the authoritative modelling region so that the new catalogue uses exactly the same spatial definition as the existing project.

In [ ]:
if not GRID_MASK_FILE.exists():
    raise FileNotFoundError(
        f"Missing {GRID_MASK_FILE}. Put california_grid_centre_mask.csv "
        "in the same folder as this notebook."
    )

grid_mask = pd.read_csv(GRID_MASK_FILE)

required_mask_cols = {"cell_id", "lat_idx", "lon_idx", "is_california"}
missing_cols = required_mask_cols - set(grid_mask.columns)
if missing_cols:
    raise ValueError(f"Grid mask is missing columns: {sorted(missing_cols)}")

assert len(grid_mask) == 1080, f"Expected 1080 retained cells, found {len(grid_mask)}"
assert grid_mask["cell_id"].is_unique, "cell_id must be unique in the grid mask"
assert grid_mask["is_california"].fillna(False).all(), "Mask contains non-California cells"

valid_cell_ids = set(grid_mask["cell_id"].astype(int))

print("Retained centre-based California cells:", len(grid_mask))
print("cell_id range:", grid_mask["cell_id"].min(), "to", grid_mask["cell_id"].max())
display(grid_mask.head())

## 2. Download USGS ComCat data (`minmagnitude = 2.0`) with caching

USGS FDSN Event Web Service limits a single query to 20,000 results, so each calendar year is downloaded with pagination. Each yearly file is cached locally. If a cache exists, rerunning this notebook loads it rather than querying the API again.

API documentation: USGS FDSN Event Web Service, `query`, `limit`, `offset`, bounding-box and magnitude parameters.

In [ ]:
USGS_QUERY_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
PAGE_LIMIT = 20000


def download_usgs_year(year, force=False, max_retries=4):
    """Download one year of the bounding-box catalogue, using pagination and a local cache."""
    year_file = RAW_DIR / f"usgs_bbox_Mge2_{year}.csv"

    if year_file.exists() and not force:
        out = pd.read_csv(year_file)
        print(f"{year}: loaded cache ({len(out):,} events)")
        return out

    session = requests.Session()
    chunks = []
    offset = 1

    while True:
        params = {
            "format": "csv",
            "starttime": f"{year}-01-01T00:00:00",
            "endtime": f"{year}-12-31T23:59:59.999",
            "minlatitude": LAT_MIN,
            "maxlatitude": LAT_MAX,
            "minlongitude": LON_MIN,
            "maxlongitude": LON_MAX,
            "minmagnitude": RAW_MIN_MAG,
            "eventtype": "earthquake",
            "orderby": "time-asc",
            "limit": PAGE_LIMIT,
            "offset": offset,
        }

        response = None
        for attempt in range(1, max_retries + 1):
            try:
                response = session.get(USGS_QUERY_URL, params=params, timeout=120)
                response.raise_for_status()
                break
            except requests.RequestException as exc:
                if attempt == max_retries:
                    raise
                wait = 2 ** (attempt - 1)
                print(f"  request failed ({exc}); retrying in {wait}s")
                time.sleep(wait)

        if response.status_code == 204 or not response.text.strip():
            break

        chunk = pd.read_csv(StringIO(response.text))
        if chunk.empty:
            break

        chunks.append(chunk)
        print(f"{year}: offset {offset:,} -> {len(chunk):,} events")

        if len(chunk) < PAGE_LIMIT:
            break

        # offset is 1-based
        offset += len(chunk)

    if not chunks:
        out = pd.DataFrame()
    else:
        out = pd.concat(chunks, ignore_index=True)

    out.to_csv(year_file, index=False)
    print(f"{year}: saved cache -> {year_file} ({len(out):,} events)")
    return out

In [ ]:
# Set FORCE_REDOWNLOAD=True only if you deliberately want to refresh all cached API data.
FORCE_REDOWNLOAD = False

if RAW_COMBINED_FILE.exists() and not FORCE_REDOWNLOAD:
    df_raw = pd.read_csv(RAW_COMBINED_FILE)
    print(f"Loaded combined raw cache: {len(df_raw):,} events")
else:
    yearly = [
        download_usgs_year(year, force=FORCE_REDOWNLOAD)
        for year in range(START_YEAR, END_YEAR + 1)
    ]
    df_raw = pd.concat(yearly, ignore_index=True)
    df_raw.to_csv(RAW_COMBINED_FILE, index=False)
    print(f"Saved combined raw catalogue: {RAW_COMBINED_FILE}")

print("Raw bounding-box events:", f"{len(df_raw):,}")
print("Raw reported magnitude range:", df_raw["mag"].min(), "to", df_raw["mag"].max())
display(df_raw.head())

## 3. Basic cleaning and audit

Only fields essential to the present modelling catalogue are required at this stage: event ID, time, location and reported magnitude. Magnitude type is **not** required for this cleaning step because unsupported or missing magnitude types are handled explicitly during harmonisation rather than silently removed here.

In [ ]:
df_clean = df_raw.copy()

# Normalise labels before any magnitude-type logic.
df_clean["magType"] = (
    df_clean["magType"].astype("string").str.strip().str.lower()
)
df_clean["magSource"] = (
    df_clean["magSource"].astype("string").str.strip().str.lower()
)
df_clean["net"] = (
    df_clean["net"].astype("string").str.strip().str.lower()
)

# Mixed timestamps occur in ComCat CSVs (some include fractional seconds, some do not).
df_clean["time"] = pd.to_datetime(
    df_clean["time"], format="mixed", utc=True, errors="coerce"
)
if "updated" in df_clean.columns:
    df_clean["updated"] = pd.to_datetime(
        df_clean["updated"], format="mixed", utc=True, errors="coerce"
    )

df_clean["year"] = df_clean["time"].dt.year

critical = ["id", "time", "latitude", "longitude", "mag"]
missing_critical = df_clean[critical].isna().any(axis=1)
invalid_coords = ~(
    df_clean["latitude"].between(-90, 90)
    & df_clean["longitude"].between(-180, 180)
)
wrong_year = ~df_clean["year"].between(START_YEAR, END_YEAR)
wrong_type = (
    df_clean["type"].ne("earthquake")
    if "type" in df_clean.columns
    else pd.Series(False, index=df_clean.index)
)

print("Raw rows:", len(df_clean))
print("Rows with missing critical fields:", int(missing_critical.sum()))
print("Rows with invalid coordinates:", int(invalid_coords.sum()))
print("Rows outside study years:", int(wrong_year.sum()))
print("Rows not labelled earthquake:", int(wrong_type.sum()))
print("Duplicate event IDs:", int(df_clean["id"].duplicated().sum()))

# Remove only observations that cannot be used reliably in the spatial/time catalogue.
df_clean = df_clean.loc[
    ~(missing_critical | invalid_coords | wrong_year | wrong_type)
].copy()

# If the service ever returns duplicate IDs, retain the latest record.
if "updated" in df_clean.columns:
    df_clean = df_clean.sort_values(["id", "updated"])
df_clean = df_clean.drop_duplicates(subset="id", keep="last").reset_index(drop=True)

print("Rows after cleaning:", f"{len(df_clean):,}")

## 4. Assign events to the established 53 × 55 grid and apply the 1,080-cell mask

The original bounding box is divided into 53 latitude intervals and 55 longitude intervals using the same `np.linspace` edges as the final project grid. Event indices are assigned using `np.searchsorted(..., side="right") - 1`, and

\[
\text{cell\_id}=i_{\text{lat}}\,n_{\text{lon}}+i_{\text{lon}}.
\]

An event is retained only if its `cell_id` occurs in the saved centre-based California mask.

In [ ]:
lat_idx = np.searchsorted(
    LAT_EDGES, df_clean["latitude"].to_numpy(), side="right"
) - 1
lon_idx = np.searchsorted(
    LON_EDGES, df_clean["longitude"].to_numpy(), side="right"
) - 1

valid_grid_index = (
    (lat_idx >= 0) & (lat_idx < N_LAT)
    & (lon_idx >= 0) & (lon_idx < N_LON)
)

# Only rows with valid indices can receive a cell ID.
df_grid = df_clean.loc[valid_grid_index].copy()
df_grid["lat_idx"] = lat_idx[valid_grid_index]
df_grid["lon_idx"] = lon_idx[valid_grid_index]
df_grid["cell_id"] = (
    df_grid["lat_idx"] * N_LON + df_grid["lon_idx"]
).astype(int)

df_grid["cell_in_mask"] = df_grid["cell_id"].isin(valid_cell_ids)
df_ca = df_grid.loc[df_grid["cell_in_mask"]].copy().reset_index(drop=True)

print("Clean bounding-box events:", f"{len(df_clean):,}")
print("Events assigned inside the 53x55 grid:", f"{len(df_grid):,}")
print("Events retained by 1,080-cell centre mask:", f"{len(df_ca):,}")
print("Occupied retained cells:", df_ca["cell_id"].nunique())

assert df_ca["cell_id"].isin(valid_cell_ids).all()
assert df_ca["cell_in_mask"].all()

df_ca.to_csv(CA_M20_FILE, index=False)
print("Saved pre-harmonisation California catalogue:", CA_M20_FILE)

## 5. Reproducibility check against the previous reported-M ≥ 2.5 catalogue

This is a diagnostic only. The rebuilt API catalogue may differ slightly from the older saved file if ComCat event metadata have been revised since the earlier download. The comparison checks whether the spatial pipeline is behaving consistently and reports any event-ID differences rather than forcing the two downloads to be identical.

In [ ]:
if REFERENCE_M25_FILE.exists():
    ref_m25 = pd.read_csv(REFERENCE_M25_FILE)
    rebuilt_reported_m25 = df_ca.loc[df_ca["mag"] >= 2.5].copy()

    ref_ids = set(ref_m25["id"].astype(str))
    rebuilt_ids = set(rebuilt_reported_m25["id"].astype(str))

    print("Old centre-mask M>=2.5 rows:", f"{len(ref_m25):,}")
    print("Rebuilt reported M>=2.5 rows:", f"{len(rebuilt_reported_m25):,}")
    print("Event IDs in both:", f"{len(ref_ids & rebuilt_ids):,}")
    print("Only in old reference:", f"{len(ref_ids - rebuilt_ids):,}")
    print("Only in rebuilt API catalogue:", f"{len(rebuilt_ids - ref_ids):,}")
else:
    print("Reference M>=2.5 catalogue not found; skipping comparison.")

## 6. Magnitude-type audit before harmonisation

The conversion rules are deliberately limited to the dominant magnitude/network combinations for which a chosen literature-supported relation is available. Other combinations remain labelled `unresolved` and are reported explicitly.

In [ ]:
magtype_summary = (
    df_ca["magType"]
    .value_counts(dropna=False)
    .rename_axis("magType")
    .reset_index(name="count")
)
magtype_summary["percentage"] = (
    magtype_summary["count"] / len(df_ca) * 100
).round(2)

display(magtype_summary)

source_summary = (
    df_ca.groupby(["magType", "net", "magSource"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(source_summary.head(30))

## 7. Harmonise supported magnitudes to an Mw-equivalent scale

### Working conversion policy

- Native moment-magnitude family (`mw`, `mwr`, `mww`, `mwb`, `mwc`): retain the reported value.
- Southern California (`ci`) `ml`: use the Southern California/Ridgecrest local-magnitude–moment relation of **Baltay & Abercrombie (2025)**, then convert seismic moment to `Mw`.
- Southern California (`ci`) `mlr`: invert the SCSN revised-local-magnitude relation to recover `ml`, then apply the same Southern California conversion.
- Northern California (`nc`) `ml`: use the central-California seismic-moment relations reported by **Bakun (1984)**.
- Northern California (`nc`) `md`: use Bakun's coda-duration relation only within its published range `1 <= Md <= 3.5`.
- Other magnitude/source combinations: leave unresolved rather than apply an unsupported conversion.

The moment-magnitude definition follows **Hanks & Kanamori (1979)**. These empirical conversions introduce uncertainty; the original magnitude and conversion method are therefore retained in the data.

In [ ]:
# ---------------------------------------------------------
# Conversion functions
# ---------------------------------------------------------

def moment_to_mw(log10_m0_nm):
    """Hanks & Kanamori (1979), with M0 in N m."""
    return (2.0 / 3.0) * (log10_m0_nm - 9.05)


def ci_ml_to_mw(ml):
    """Southern California ML -> Mw-equivalent; Baltay & Abercrombie (2025)."""
    ml = np.asarray(ml, dtype=float)
    log10_m0_nm = (
        1.5 * ml
        + 0.5 * np.logaddexp(0.0, 4.6 - ml)
        + 8.35
    )
    return moment_to_mw(log10_m0_nm)


def ci_mlr_to_mw(mlr):
    """Invert SCSN MLr = 0.853 ML + 0.40125, then use CI ML conversion."""
    mlr = np.asarray(mlr, dtype=float)
    ml = (mlr - 0.40125) / 0.853
    return ci_ml_to_mw(ml)


def nc_ml_to_mw(ml):
    """Central/Northern California ML -> Mw-equivalent using Bakun (1984)."""
    ml = np.asarray(ml, dtype=float)

    # Bakun reports overlapping magnitude-dependent relations.
    # We use the lower-magnitude relation through ML=3.5 and the
    # higher-magnitude relation above 3.5.
    log10_m0_dyn_cm = np.where(
        ml <= 3.5,
        1.2 * ml + 17.0,
        1.5 * ml + 16.0,
    )

    # 1 N m = 10^7 dyne cm
    log10_m0_nm = log10_m0_dyn_cm - 7.0
    return moment_to_mw(log10_m0_nm)


def nc_md_to_mw(md):
    """Northern California Md -> Mw-equivalent; Bakun (1984), valid for 1<=Md<=3.5."""
    md = np.asarray(md, dtype=float)
    log10_m0_dyn_cm = 1.2 * md + 17.0
    log10_m0_nm = log10_m0_dyn_cm - 7.0
    return moment_to_mw(log10_m0_nm)

In [ ]:
df_h = df_ca.copy()

df_h["mag_original"] = df_h["mag"]
df_h["magType_original"] = df_h["magType"]
df_h["mag_Mw"] = np.nan
df_h["mag_conversion"] = "unresolved"

# Native moment-magnitude family
native_mw_types = {"mw", "mwr", "mww", "mwb", "mwc"}
native_mw = df_h["magType"].isin(native_mw_types)
df_h.loc[native_mw, "mag_Mw"] = df_h.loc[native_mw, "mag"]
df_h.loc[native_mw, "mag_conversion"] = "native_Mw_family"

# Southern California ML
ci_ml = (df_h["magType"] == "ml") & (df_h["magSource"] == "ci")
df_h.loc[ci_ml, "mag_Mw"] = ci_ml_to_mw(df_h.loc[ci_ml, "mag"])
df_h.loc[ci_ml, "mag_conversion"] = "BaltayAbercrombie2025_CI_ML"

# Southern California revised ML
ci_mlr = (df_h["magType"] == "mlr") & (df_h["magSource"] == "ci")
df_h.loc[ci_mlr, "mag_Mw"] = ci_mlr_to_mw(df_h.loc[ci_mlr, "mag"])
df_h.loc[ci_mlr, "mag_conversion"] = "SCEDC_MLr_then_Baltay2025"

# Northern California ML
nc_ml = (df_h["magType"] == "ml") & (df_h["magSource"] == "nc")
df_h.loc[nc_ml, "mag_Mw"] = nc_ml_to_mw(df_h.loc[nc_ml, "mag"])
df_h.loc[nc_ml, "mag_conversion"] = "Bakun1984_NC_ML"

# Northern California MD, within published range only
nc_md_valid = (
    (df_h["magType"] == "md")
    & (df_h["magSource"] == "nc")
    & df_h["mag"].between(1.0, 3.5)
)
df_h.loc[nc_md_valid, "mag_Mw"] = nc_md_to_mw(df_h.loc[nc_md_valid, "mag"])
df_h.loc[nc_md_valid, "mag_conversion"] = "Bakun1984_NC_MD"

nc_md_outside = (
    (df_h["magType"] == "md")
    & (df_h["magSource"] == "nc")
    & ~df_h["mag"].between(1.0, 3.5)
)
df_h.loc[nc_md_outside, "mag_conversion"] = "MD_outside_Bakun_range"

print("Magnitude harmonisation applied.")

## 8. Audit conversion coverage before applying any final threshold

This checkpoint is important: unresolved observations are not silently converted or silently ignored. Their magnitude type and reporting source are shown before the final catalogue is constructed.

In [ ]:
conversion_summary = (
    df_h["mag_conversion"]
    .value_counts(dropna=False)
    .rename_axis("method")
    .reset_index(name="count")
)
conversion_summary["percentage"] = (
    conversion_summary["count"] / len(df_h) * 100
).round(2)

display(conversion_summary)

unresolved = df_h.loc[df_h["mag_Mw"].isna()].copy()
unresolved_summary = (
    unresolved.groupby(["magType", "net", "magSource"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(unresolved_summary)

unresolved_pct = 100 * len(unresolved) / len(df_h)
print(f"Unresolved: {len(unresolved):,} / {len(df_h):,} ({unresolved_pct:.2f}%)")
print("NC Md outside Bakun range:", int(nc_md_outside.sum()))

if unresolved_pct > 1.0:
    warnings.warn(
        "More than 1% of the centre-mask catalogue is unresolved. "
        "Review unresolved_summary before treating the final Mw>=2.5 catalogue as fixed."
    )

## 9. Quantify the effect of harmonisation on the 2.5 and 3.0 thresholds

The first comparison answers whether the wider M ≥ 2.0 download was necessary: it counts events with reported magnitude below 2.5 that move to `Mw >= 2.5` after harmonisation. The second comparison records how many observations change classification around 3.0; this will be used later when deciding and justifying the forecasting target threshold.

In [ ]:
resolved = df_h["mag_Mw"].notna()

added_across_25 = (
    resolved
    & (df_h["mag_original"] < 2.5)
    & (df_h["mag_Mw"] >= 2.5)
)
lost_across_25 = (
    resolved
    & (df_h["mag_original"] >= 2.5)
    & (df_h["mag_Mw"] < 2.5)
)

print("Reported M<2.5 but harmonised Mw>=2.5:", int(added_across_25.sum()))
print("Reported M>=2.5 but harmonised Mw<2.5:", int(lost_across_25.sum()))

threshold_25 = pd.crosstab(
    df_h.loc[resolved, "mag_original"] >= 2.5,
    df_h.loc[resolved, "mag_Mw"] >= 2.5,
    rownames=["Original reported M >= 2.5"],
    colnames=["Harmonised Mw >= 2.5"],
)
display(threshold_25)

threshold_30 = pd.crosstab(
    df_h.loc[resolved, "mag_original"] >= 3.0,
    df_h.loc[resolved, "mag_Mw"] >= 3.0,
    rownames=["Original reported M >= 3.0"],
    colnames=["Harmonised Mw >= 3.0"],
)
display(threshold_30)

m3_changed = (
    (df_h.loc[resolved, "mag_original"] >= 3.0)
    != (df_h.loc[resolved, "mag_Mw"] >= 3.0)
)
print("Events changing M=3.0 classification:", int(m3_changed.sum()))
print("Percentage of resolved events:", f"{100*m3_changed.mean():.2f}%")

## 10. Construct and save the final Mw ≥ 2.5 modelling catalogue

Events without a supported magnitude conversion are saved separately. The working final catalogue contains only observations with a resolved `Mw`-equivalent magnitude satisfying `Mw >= 2.5`.

This exclusion rule must be reported together with the unresolved count and percentage. If the unresolved proportion is materially larger than expected, revisit the conversion policy before modelling.

In [ ]:
# Save every centre-mask event with the conversion fields for full traceability.
df_h.to_csv(HARMONISED_ALL_FILE, index=False)
unresolved.to_csv(UNRESOLVED_FILE, index=False)

# Final input catalogue: supported harmonisation + Mw >= 2.5.
df_final = df_h.loc[
    df_h["mag_Mw"].notna() & (df_h["mag_Mw"] >= 2.5)
].copy().reset_index(drop=True)

# Keep a clear working magnitude variable while preserving all original columns.
df_final["mag_model"] = df_final["mag_Mw"]

df_final.to_csv(FINAL_MW25_FILE, index=False)

print("Saved all harmonised/flagged centre-mask events:", HARMONISED_ALL_FILE)
print("Saved unresolved events:", UNRESOLVED_FILE)
print("Saved final Mw>=2.5 catalogue:", FINAL_MW25_FILE)
print()
print("Final events:", f"{len(df_final):,}")
print("Occupied cells:", df_final["cell_id"].nunique())
print("Full modelling domain remains:", len(grid_mask), "cells")
print("Magnitude range (Mw-equivalent):", df_final["mag_Mw"].min(), "to", df_final["mag_Mw"].max())

## 11. Save audit tables for the report

These small CSVs make it easy to quote exact numbers later without rerunning the full notebook.

In [ ]:
conversion_summary.to_csv(DATA_DIR / "audit_magnitude_conversion_summary.csv", index=False)
unresolved_summary.to_csv(DATA_DIR / "audit_magnitude_unresolved_summary.csv", index=False)
threshold_25.to_csv(DATA_DIR / "audit_threshold_25_crossing.csv")
threshold_30.to_csv(DATA_DIR / "audit_threshold_30_crossing.csv")
magtype_summary.to_csv(DATA_DIR / "audit_magtype_summary_Mge2_centre_mask.csv", index=False)
source_summary.to_csv(DATA_DIR / "audit_magtype_network_source_Mge2_centre_mask.csv", index=False)

print("Audit tables saved.")

## 12. Set the rebuilt catalogue as the working `df`

From this point onward, subsequent EDA, daily grid counts, predictor construction and target construction should use the harmonised `Mw >= 2.5` catalogue below. The target magnitude threshold should **not** be hard-coded until the planned threshold sensitivity check is completed.

In [ ]:
df = df_final.copy()

print("Working df:", df.shape)
print("Dates:", df["time"].min(), "to", df["time"].max())
print("Grid cells with at least one event:", df["cell_id"].nunique())
display(df.head())